In [1]:
import json
import numpy as np
import pandas as pd

DATA_PATH = "arData/evaluation.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

len(data)

8

In [2]:
def calculate_sus_score(form_data):
    """Berechnet den System Usability Scale (SUS) Score."""
    score = 0
    # Ungerade Fragen: Nutzer-Score - 1
    score += (form_data.get("usability1", 1) - 1)
    score += (form_data.get("usability3", 1) - 1)
    score += (form_data.get("usability5", 1) - 1)
    score += (form_data.get("usability7", 1) - 1)
    score += (form_data.get("usability9", 1) - 1)
    
    # Gerade Fragen: 5 - Nutzer-Score
    score += (5 - form_data.get("usability2", 5))
    score += (5 - form_data.get("usability4", 5))
    score += (5 - form_data.get("usability6", 5))
    score += (5 - form_data.get("usability8", 5))
    score += (5 - form_data.get("usability10", 5))
    
    return score * 2.5

# Definition der Fragetypen basierend auf dem HARUS-Paper
aPOSITIV = [2, 4, 6, 8, 10, 14, 16]
MANIP_QS = list(range(1, 9))
COMPR_QS = list(range(9, 17))

def calculate_harus_total(form_data, positively_stated=aPOSITIV):
    """Berechnet den gesamten HARUS-Score auf 0..100 (Summe/0.96)."""
    converted_scores = []
    for i in range(1, 17):
        key = f"har{i}"
        score = form_data.get(key, 4)  # neutral 4
        if i in positively_stated:
            converted = score - 1       # 0..6
        else:
            converted = 7 - score       # 0..6
        converted_scores.append(converted)
    total_score = sum(converted_scores)  # 0..96
    return total_score / 0.96            # 0..100

def calculate_harus_sub(form_data, questions, positively_stated=aPOSITIV):
    """Berechnet Subskalen (0..100) für gegebene Frage-IDs."""
    converted_scores = []
    for i in questions:
        key = f"har{i}"
        score = form_data.get(key, 4)
        if i in positively_stated:
            converted = score - 1
        else:
            converted = 7 - score
        converted_scores.append(converted)
    return (sum(converted_scores) / (len(questions) * 6)) * 100

In [3]:
def _is_null_viewer_position(vp):
    # Null-Erkennung robust: None, "null", leere Liste, Liste mit None-Werten
    if vp is None:
        return True
    if isinstance(vp, str) and vp.strip().lower() == "null":
        return True
    if isinstance(vp, (list, tuple)):
        if len(vp) == 0:
            return True
        return all(x is None for x in vp)
    return False

def time_to_place_from_viewer_position(metrics):
    """Sekunden bis zur ersten viewerPosition ≠ null; Start = frühester Timestamp der Session."""
    # t0 = frühester Timestamp aus allen Streams
    all_ts = []
    for key in ("arTrackingData", "deviceOrientations", "interactions"):
        for e in metrics.get(key, []):
            if isinstance(e, dict) and "timestamp" in e:
                all_ts.append(e["timestamp"])
    if not all_ts:
        return None
    t0 = min(all_ts)

    # erstes non-null viewerPosition in arTrackingData
    ar = sorted(metrics.get("arTrackingData", []), key=lambda x: x.get("timestamp", 0))
    for e in ar:
        if not _is_null_viewer_position(e.get("viewerPosition")):
            return (e["timestamp"] - t0) / 1000.0
    return None

def session_time_from_ar_tracking(metrics):
    """Gesamtdauer der Session in Sekunden auf Basis von arTrackingData (min(ts) -> max(ts))."""
    ar = metrics.get("arTrackingData", [])
    ts = [e.get("timestamp") for e in ar
          if isinstance(e, dict) and isinstance(e.get("timestamp"), (int, float))]
    if len(ts) < 2:
        # Keine Dauer berechenbar (0 oder 1 Sample)
        return None, None, None
    t0 = int(min(ts))
    t1 = int(max(ts))
    return (t1 - t0) / 1000.0, t0, t1  # (Sekunden, Start(ms), Ende(ms))


def tracking_metrics(metrics):
    """Grobe Tracking-Stabilität: Uptime (%) via Median-Sample-Dt, sowie Dropouts/min (>1s Lücken)."""
    ar = metrics.get("arTrackingData", [])
    ts = sorted([e["timestamp"] for e in ar if "timestamp" in e])
    if len(ts) < 2:
        return None, None
    duration = (ts[-1] - ts[0]) / 1000.0
    gaps = [(ts[i] - ts[i-1]) / 1000.0 for i in range(1, len(ts))]
    median_dt = float(np.median(gaps)) if gaps else 0.1
    covered = (len(ts) - 1) * median_dt
    uptime_pct = (covered / duration * 100.0) if duration > 0 else None
    dropouts = sum(1 for g in gaps if g > 1.0)
    drop_per_min = (dropouts / (duration / 60.0)) if duration > 0 else None
    return uptime_pct, drop_per_min

def slider_metrics(metrics):
    ints = metrics.get("interactions", [])
    vals = [float(e["value"]) for e in ints if e.get("elementId") == "offset-slider" and e.get("type") == "input" and e.get("value") not in (None, "")]
    if not vals:
        return 0, None, None, None, None
    return len(vals), min(vals), max(vals), (max(vals) - min(vals)), vals[-1]

In [4]:
rows = []
for p in data:
    fd = p.get("formData", {})
    metrics = p.get("metrics", {})

    sus = calculate_sus_score(fd)
    harus_total = calculate_harus_total(fd)
    harus_manip = calculate_harus_sub(fd, MANIP_QS)
    harus_comp = calculate_harus_sub(fd, COMPR_QS)

    ttp = time_to_place_from_viewer_position(metrics)
    uptime, drop_pm = tracking_metrics(metrics)

    moves, off_min, off_max, off_range, off_final = slider_metrics(metrics)

    sess_dur_s, sess_start_ms, sess_end_ms = session_time_from_ar_tracking(metrics)

    rows.append({
        "deviceId": p.get("deviceId"),
        "SUS": round(sus, 2),
        "HARUS_total (0–100)": round(harus_total, 2),
        "HARUS_Manipulierbarkeit (0–100)": round(harus_manip, 2),
        "HARUS_Verständlichkeit (0–100)": round(harus_comp, 2),
        "Time-to-Place_viewerPosition (s)": round(ttp, 3) if ttp is not None else None,
        "Tracking-Uptime (%)": round(uptime, 3) if uptime is not None else None,
        "Dropouts/min": round(drop_pm, 3) if drop_pm is not None else None,
        "Slider-Bewegungen": moves,
        "Offset-Min": off_min,
        "Offset-Max": off_max,
        "Offset-Range": off_range,
        "Offset-Final": off_final,

        # NEU: Session-Infos
        "Session-Start (ms)": sess_start_ms,
        "Session-Ende (ms)": sess_end_ms,
        "Session-Dauer (s)": round(sess_dur_s, 3) if sess_dur_s is not None else None,
    })

per_user = pd.DataFrame(rows)
per_user


,deviceId,SUS,HARUS_total (0–100),HARUS_Manipulierbarkeit (0–100),HARUS_Verständlichkeit (0–100),Time-to-Place_viewerPosition (s),Tracking-Uptime (%),Dropouts/min,Slider-Bewegungen,Offset-Min,Offset-Max,Offset-Range,Offset-Final,Session-Start (ms),Session-Ende (ms),Session-Dauer (s)
0,0bddb404-39fa-45ab-93c0-a1085904c2d8,82.5,72.92,79.17,66.67,1.400,99.735,0.0,124,0.09,1.24,1.15,0.64,1758811016335,1758811054135,37.800
1,c5bd2408-5565-490b-8556-4b6a83758e8f,90.0,88.54,95.83,81.25,1.700,99.631,0.0,107,0.10,1.47,1.37,1.47,1758811094461,1758811121561,27.100
2,69d49594-202f-4aa2-ab20-b300972bbaf0,100.0,92.71,85.42,100.00,1.199,100.002,0.0,368,0.08,1.65,1.57,1.17,1758820270512,1758820318811,48.299
3,c3987e23-d978-432b-8e23-880b44f031f7,87.5,71.88,87.50,56.25,1.100,99.998,0.0,339,-0.05,2.00,2.05,0.80,1758820261755,1758820308756,47.001
4,8830974d-583e-4f6b-a143-47945d96be16,82.5,73.96,66.67,81.25,1.802,99.346,0.0,107,0.07,0.90,0.83,0.35,1758877629430,1758877660634,31.204
5,a8bab09e-2947-4f9d-84b0-4dcd6b079079,75.0,82.29,91.67,72.92,2.400,99.608,0.0,49,0.09,0.76,0.67,0.76,1758890519131,1758890595631,76.500
6,2987226a-fbf8-483d-a836-f2181d3ded46,90.0,79.17,83.33,75.00,0.999,100.000,0.0,73,-0.09,0.78,0.87,0.68,1758892478800,1758892507500,28.700
7,aac9452b-9f32-4630-9ded-c79a403c9a40,90.0,88.54,95.83,81.25,0.800,99.775,0.0,198,-0.12,0.88,1.00,0.63,1758981198682,1758981243082,44.400


In [7]:
med = {
    "N": len(per_user),
    "SUS_Median": round(float(per_user["SUS"].median()), 2),
    "HARUS_total_Median": round(float(per_user["HARUS_total (0–100)"].median()), 2),
    "HARUS_Manipulierbarkeit_Median": round(float(per_user["HARUS_Manipulierbarkeit (0–100)"].median()), 2),
    "HARUS_Verständlichkeit_Median": round(float(per_user["HARUS_Verständlichkeit (0–100)"].median()), 2),
    "Time-to-Place_Median (s)": round(float(per_user["Time-to-Place_viewerPosition (s)"].median()), 3),
    "Tracking-Uptime_Median (%)": round(float(per_user["Tracking-Uptime (%)"].median()), 3),
    "Dropouts/min_Median": round(float(per_user["Dropouts/min"].median()), 3),
    "Slider-Bewegungen_Median": int(per_user["Slider-Bewegungen"].median()) if len(per_user) else 0,
    "Offset_Final": round(float(per_user["Offset-Final"].median()), 3),
    "Session-Dauer_Median (s)": round(float(per_user["Session-Dauer (s)"].mean()), 3),
}
med_df = pd.DataFrame([med])
med_df


,N,SUS_Median,HARUS_total_Median,HARUS_Manipulierbarkeit_Median,HARUS_Verständlichkeit_Median,Time-to-Place_Median (s),Tracking-Uptime_Median (%),Dropouts/min_Median,Slider-Bewegungen_Median,Offset_Final,Session-Dauer_Median (s)
0,8,88.75,80.73,86.46,78.12,1.3,99.755,0.0,115,0.72,42.626


In [6]:
out_csv = "study_summary_updated.csv"
per_user.to_csv(out_csv, index=False)
out_csv

'study_summary_updated.csv'